# 🌳 Refauna SLM — Gemma 4 E2B com **LoRA 16-bit** (mesmos dados do QLoRA)

Terceira rodada do experimento da palestra: **o mesmo Gemma 4 E2B, os mesmos dados, o mesmo prompt e o mesmo juiz** — muda só a forma de carregar o modelo-base durante o treino.

| | 1ª rodada (já publicada) | **Esta rodada** |
|---|---|---|
| Modelo-base no treino | comprimido em **4 bits** (QLoRA) | em **16 bits** (LoRA), sem compressão |
| Adaptadores LoRA | r=16, atenção + MLP | **idênticos** |
| Dados, prompt, hiperparâmetros, avaliação | — | **idênticos** (lidos do Drive) |

**Pergunta que esta rodada responde:** quanto de qualidade o QLoRA "custa" (se custa) em troca da economia de memória? A tabela final junta **Gemma QLoRA × Gemma LoRA × Qwen LoRA**.

**Pré-requisito:** `refauna-slm/data/` no Drive com `chunks.jsonl`, `chunk_embeddings.npy`, `docs.jsonl` e `sft/{train,val,test}.jsonl` (gerados pelo notebook original do Gemma). Mesmos Secrets do Colab (`HF_TOKEN`, `ANTHROPIC_API_KEY`). GPU: L4 ou A100 (o LoRA 16-bit do E2B usa ~8–10 GB).

**Nada é sobrescrito:** resultados vão para `outputs/gemma4-e2b-refauna-lora16/` e a publicação usa repositórios novos (`…-lora16…`).

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q unsloth
!pip install -q chromadb rank-bm25 "sentence-transformers>=5.0" anthropic

import importlib.util
faltando = [m for m in ["unsloth", "chromadb", "rank_bm25", "sentence_transformers", "anthropic"]
            if importlib.util.find_spec(m) is None]
print("✅ todas as bibliotecas instaladas" if not faltando else f"❌ faltando: {faltando}")

> Se o Colab pedir para **reiniciar a sessão**, reinicie e continue a partir da célula abaixo (sem reinstalar).

In [ ]:
import unsloth  # importe a Unsloth antes de transformers/trl
import os, re, json, random, time, shutil, unicodedata, csv
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from google.colab import drive, userdata

drive.mount("/content/drive")

# ======================= CONFIGURAÇÃO =======================
HF_NAMESPACE = "Refauna"        # organização onde estão os modelos do Gemma
HF_PRIVATE   = True             # publica privado; revise o model card e torne público no site

PROJECT_DIR  = Path("/content/drive/MyDrive/refauna-slm")
if not (PROJECT_DIR / "training-files").exists():
    found = sorted(Path("/content/drive/MyDrive").glob("**/training-files"))
    assert found, "Pasta 'training-files' não encontrada no Meu Drive. Ajuste PROJECT_DIR."
    PROJECT_DIR = found[0].parent
DATA_DIR  = PROJECT_DIR / "data"
OUT_DIR   = PROJECT_DIR / "outputs"
GOLD_FILE = PROJECT_DIR / "gold_questions.jsonl"
LOCAL_DIR = Path("/content/work"); LOCAL_DIR.mkdir(parents=True, exist_ok=True)

for f in ["chunks.jsonl", "docs.jsonl", "sft/train.jsonl", "sft/val.jsonl", "sft/test.jsonl"]:
    assert (DATA_DIR / f).exists(), f"Falta {DATA_DIR/f}. Rode antes o notebook original do Gemma até a seção 5."

BASE_MODEL   = "unsloth/gemma-4-E2B-it"        # o MESMO da 1ª rodada
EMBED_MODEL  = "google/embeddinggemma-300m"
GEN_MODEL    = "claude-sonnet-5-5"             # juiz da avaliação (o mesmo das outras rodadas)

LOAD_IN_4BIT = False   # <<< a ÚNICA mudança em relação à 1ª rodada: LoRA 16-bit em vez de QLoRA
MAX_SEQ_LEN  = 4096
LORA_R       = 16
EPOCHS       = 2
LR           = 2e-4
TOP_K        = 4
SEED         = 3407
RUN_NAME     = "gemma4-e2b-refauna-lora16"

# Repositórios NOVOS (não sobrescrevem os do QLoRA: gemma-4-e2b-refauna, -lora, -GGUF)
REPO_LORA   = f"{HF_NAMESPACE}/gemma-4-e2b-refauna-lora16-adapter"
REPO_MERGED = f"{HF_NAMESPACE}/gemma-4-e2b-refauna-lora16"
REPO_GGUF   = f"{HF_NAMESPACE}/gemma-4-e2b-refauna-lora16-GGUF"

assert not (OUT_DIR / RUN_NAME / "train_report.json").exists() or input(
    f"Já existe uma rodada '{RUN_NAME}' em outputs/. Digite 'sim' para sobrescrever: ") == "sim"

random.seed(SEED)
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
from huggingface_hub import login
login(token=HF_TOKEN)
print("GPU:", torch.cuda.get_device_name(0), "| projeto:", PROJECT_DIR, "| run:", RUN_NAME)

In [ ]:
def write_jsonl(path, rows):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

### Contrato de prompt — idêntico ao das outras rodadas

In [ ]:
SYSTEM_PROMPT = (
    "Você é o Assistente Refauna, especialista em defaunação, refaunação, reintrodução de fauna, "
    "dispersão de sementes, interações ecológicas e conservação da Mata Atlântica. "
    "Responda sempre em português do Brasil, de forma clara, didática e precisa. "
    "Quando houver CONTEXTO, use somente as informações dele e cite os trechos usados no formato [n]. "
    "Se o CONTEXTO não contiver a resposta, diga explicitamente que não encontrou essa informação "
    "nos documentos do Refauna e não invente dados, números, datas ou nomes de espécies."
)

ABSTAIN_ANSWERS = [
    "Não encontrei essa informação nos documentos do Refauna fornecidos no contexto. Os trechos disponíveis tratam de outros aspectos e não permitem responder com segurança.",
    "Os trechos recuperados não contêm a resposta para essa pergunta. Prefiro não especular: não encontrei essa informação nos documentos do Refauna.",
    "Com base no contexto fornecido, não é possível responder. Não encontrei nos documentos do Refauna dados sobre esse ponto específico.",
    "Não encontrei essa informação nos documentos do Refauna. O contexto disponível aborda temas relacionados, mas não traz esse dado.",
    "Essa informação não aparece nos trechos dos documentos do Refauna que recebi, então não consigo responder sem correr o risco de inventar.",
]
ABSTAIN_RE = re.compile(r"não (encontrei|consigo responder|é possível responder|há informaç|aparece)|não contêm a resposta", re.I)

def is_abstention(text):
    return bool(ABSTAIN_RE.search(text or ""))

NAME_SUFFIX = {"jr", "júnior", "junior", "filho", "neto", "sobrinho"}

def short_ref(c):
    """'(Cid et al., 2014 — Título curto)'. Aceita autores separados por ';' (planilha) ou ','."""
    raw = (c.get("authors") or "").strip()
    has_etal = bool(re.search(r"\bet al\b", raw))
    raw = re.sub(r"\bet al\.?", "", raw).strip(" ;,")
    parts = [a.strip() for a in re.split(r";" if ";" in raw else r",| & | and ", raw) if a.strip()]
    first = parts[0].split(",")[0] if parts else ""          # "Fernandez, F." -> "Fernandez"
    words = [w for w in first.split() if w.rstrip(".").lower() not in NAME_SUFFIX]
    surname = words[-1] if words else ""
    year = c.get("year") or "s.d."
    title = (c.get("title") or c.get("doc_id", "")).strip()
    title = title if len(title) <= 60 else title[:57].rstrip() + "..."
    who = (f"{surname} et al., " if (len(parts) > 1 or has_etal) else f"{surname}, ") if surname else ""
    return f"({who}{year} — {title})"

def format_context(chunks):
    return "\n".join(f"[{i}] {short_ref(c)} {c['text'].strip()}" for i, c in enumerate(chunks, 1))

def build_user_message(question, chunks=None):
    if chunks:
        return f"CONTEXTO:\n{format_context(chunks)}\n\nPERGUNTA: {question.strip()}"
    return question.strip()

def build_messages(question, chunks=None):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_message(question, chunks)}]

PT_SW = set("de que não para com uma os as dos das em é são também pela pelo mais como sua seu ao".split())
EN_SW = set("the and of to is are with for that this which were from by their".split())
def is_ptbr(text):
    toks = re.findall(r"\w+", (text or "").lower())
    return sum(t in PT_SW for t in toks) >= sum(t in EN_SW for t in toks)

## 1. Dados e índice (reaproveitados do Drive)
Mesmos trechos, mesmos embeddings, mesmo split. O índice híbrido é remontado em memória.

In [ ]:
docs   = read_jsonl(DATA_DIR / "docs.jsonl")
chunks = read_jsonl(DATA_DIR / "chunks.jsonl")
chunk_by_id = {c["chunk_id"]: c for c in chunks}
train, val, test = (read_jsonl(DATA_DIR / "sft" / f"{s}.jsonl") for s in ("train", "val", "test"))
print(f"{len(docs)} documentos | {len(chunks)} trechos | train={len(train)} val={len(val)} test={len(test)}")
print("Composição do treino:", dict(Counter(e["kind"] for e in train)))

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

embedder = SentenceTransformer(EMBED_MODEL, device="cuda", model_kwargs={"torch_dtype": torch.float32})

def index_text(c):
    return f"{c['title']} — {c['section']}\n{c['text']}"

emb_path = DATA_DIR / "chunk_embeddings.npy"
doc_emb = np.load(emb_path) if emb_path.exists() else None
if doc_emb is None or len(doc_emb) != len(chunks):
    print("Embeddings salvos ausentes ou desatualizados — recalculando...")
    doc_emb = embedder.encode_document([index_text(c) for c in chunks], batch_size=32, normalize_embeddings=True)

chroma = chromadb.EphemeralClient()
col = chroma.create_collection("refauna", metadata={"hnsw:space": "cosine"})
for i in range(0, len(chunks), 500):
    b = chunks[i:i + 500]
    col.add(ids=[c["chunk_id"] for c in b], embeddings=doc_emb[i:i + 500].tolist(), documents=[c["text"] for c in b])
print("Índice denso:", col.count(), "vetores")

In [ ]:
STOP = set(("a o e de da do das dos em no na nos nas um uma para por com que se ao os as sao "
            "the of and to in is for on with by as are was were be this that from at an or").split())

def bm25_tokens(s):
    s = unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in STOP and len(t) > 1]

def rrf_fuse(rankings, rrf_k=60):
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, cid in enumerate(ranking):
            fused[cid] += 1.0 / (rrf_k + rank + 1)
    return sorted(fused, key=fused.get, reverse=True)

def select_diverse(ranked_ids, by_id, k, exclude_ids=()):
    """Remove excluídos e chunks vizinhos (sobrepostos) do mesmo documento."""
    out = []
    for cid in ranked_ids:
        if cid in exclude_ids:
            continue
        c = by_id[cid]
        if any(o["doc_id"] == c["doc_id"] and abs(o["idx"] - c["idx"]) <= 1 for o in out):
            continue
        out.append(c)
        if len(out) == k:
            break
    return out

In [ ]:
bm25 = BM25Okapi([bm25_tokens(index_text(c)) for c in chunks])
chunk_ids = [c["chunk_id"] for c in chunks]

def hybrid_search(query, k=TOP_K, n_candidates=20, exclude_ids=()):
    q = embedder.encode_query(query, normalize_embeddings=True)
    dense = col.query(query_embeddings=[q.tolist()], n_results=min(n_candidates, len(chunks)))["ids"][0]
    scores = bm25.get_scores(bm25_tokens(query))
    lexical = [chunk_ids[i] for i in np.argsort(scores)[::-1][:n_candidates]]
    return select_diverse(rrf_fuse([dense, lexical]), chunk_by_id, k, exclude_ids)

for q in ["O que é refaunação e por que ela importa para a Mata Atlântica?",
          "Quais animais foram reintroduzidos no Parque Nacional da Tijuca?",
          "Qual o papel das cutias na dispersão de sementes?"]:
    print(f"\n❓ {q}")
    for i, c in enumerate(hybrid_search(q), 1):
        print(f"  [{i}] {short_ref(c)} · {c['section']} · p.{c['page_start']} :: {c['text'][:110]}...")

## 2. Fine-tuning do Gemma 4 E2B com LoRA 16-bit
Células copiadas da 1ª rodada. Com `LOAD_IN_4BIT = False`, o modelo-base é carregado em 16 bits e congelado; os adaptadores LoRA (r=16, atenção + MLP, só camadas de linguagem) são os mesmos.

> ℹ️ Loss inicial na faixa de **13–15** continua normal no Gemma 4 E2B. Compare a **curva** e a `eval_loss` com a da rodada QLoRA (`outputs/gemma4-e2b-refauna-qlora/loss.png`) — aqui a comparação de loss **é** válida, porque o modelo e o tokenizer são os mesmos.

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL, dtype=None, max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT, full_finetuning=False, token=HF_TOKEN,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none", random_state=SEED,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
tok = getattr(tokenizer, "tokenizer", tokenizer)   # tokenizer de texto (se vier um processor multimodal)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {trainable/1e6:.1f}M de {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

In [ ]:
from datasets import Dataset

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
                     for m in batch["messages"]]}

def build_ds(rows):
    ds = Dataset.from_list([{"messages": e["messages"]} for e in rows]).map(to_text, batched=True, remove_columns=["messages"])
    n0 = len(ds)
    ds = ds.filter(lambda x: len(tok(x["text"], add_special_tokens=False)["input_ids"]) <= MAX_SEQ_LEN)
    if len(ds) < n0:
        print(f"  {n0-len(ds)} exemplos acima de {MAX_SEQ_LEN} tokens removidos")
    return ds

train_ds, val_ds = build_ds(train), build_ds(val)
print(f"train={len(train_ds)} val={len(val_ds)}\n")
print("Como o Gemma 4 enxerga um exemplo de treino:\n")
print(train_ds[0]["text"][:1500])

In [ ]:
from trl import SFTTrainer, SFTConfig

CKPT_DIR = LOCAL_DIR / RUN_NAME / "checkpoints"
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS, learning_rate=LR, warmup_ratio=0.05,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.001,
        logging_steps=5, eval_strategy="steps", eval_steps=50,
        save_strategy="steps", save_steps=50, save_total_limit=3,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        seed=SEED, output_dir=str(CKPT_DIR), report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|turn>user\n", response_part="<|turn>model\n")

# Verificação do mascaramento: deve sobrar só a resposta do modelo
labels = trainer.train_dataset[0]["labels"]
print(tok.decode([t for t in labels if t != -100])[:600])

In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer_stats = trainer.train()
train_minutes = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_reserved() / 1024**3

train_report = {
    "run": RUN_NAME, "load_in_4bit": LOAD_IN_4BIT, "lora_r": LORA_R, "epochs": EPOCHS, "lr": LR,
    "train_examples": len(train_ds), "minutes": round(train_minutes, 1), "peak_vram_gb": round(peak_gb, 2),
    "trainable_params_m": round(trainable / 1e6, 1), "gpu": torch.cuda.get_device_name(0),
    "best_eval_loss": trainer.state.best_metric,
}
(OUT_DIR / RUN_NAME).mkdir(parents=True, exist_ok=True)
json.dump(train_report, open(OUT_DIR / RUN_NAME / "train_report.json", "w"), indent=2)
json.dump(trainer.state.log_history, open(OUT_DIR / RUN_NAME / "log_history.json", "w"))
print(json.dumps(train_report, indent=2))

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(8, 4))
plt.plot(*zip(*tr), label="train loss", alpha=.7)
if ev:
    plt.plot(*zip(*ev), "o-", label="eval loss")
plt.xlabel("passo"); plt.ylabel("loss"); plt.title(f"Gemma 4 E2B · {RUN_NAME}"); plt.legend(); plt.grid(alpha=.3)
plt.savefig(OUT_DIR / RUN_NAME / "loss.png", dpi=160, bbox_inches="tight"); plt.show()

ADAPTER_DIR = OUT_DIR / RUN_NAME / "lora"
model.save_pretrained(str(ADAPTER_DIR)); tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Adapter salvo em", ADAPTER_DIR)

In [ ]:
# Curvas de loss: QLoRA (1ª rodada) × LoRA 16-bit (esta rodada) — mesmo modelo, mesmo tokenizer
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
for run, estilo in [("gemma4-e2b-refauna-qlora", "--"), (RUN_NAME, "-")]:
    p = OUT_DIR / run / "log_history.json"
    if not p.exists():
        print("sem log_history para", run); continue
    hist = json.load(open(p))
    ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    if ev:
        ax.plot(*zip(*ev), estilo, marker="o", label=f"{run} (eval)")
ax.set_xlabel("passo"); ax.set_ylabel("eval loss"); ax.set_title("Gemma 4 E2B · QLoRA × LoRA 16-bit")
ax.grid(alpha=.3); ax.legend()
plt.savefig(OUT_DIR / RUN_NAME / "loss_qlora_vs_lora16.png", dpi=160, bbox_inches="tight"); plt.show()

### RAG ponta a ponta no Colab

In [ ]:
FastModel.for_inference(model)

def _apply(messages):
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                         return_dict=True, return_tensors="pt")

def generate(messages, max_new_tokens=512, temperature=0.3):
    try:
        inputs = _apply(messages)
    except Exception:   # alguns processors exigem conteúdo no formato multimodal
        inputs = _apply([{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages])
    inputs = inputs.to("cuda")
    kw = dict(max_new_tokens=max_new_tokens, use_cache=True)   # use_cache=True é obrigatório no Gemma 4 E2B
    kw.update(do_sample=True, temperature=temperature, top_p=0.95, top_k=64) if temperature > 0 else kw.update(do_sample=False)
    with torch.inference_mode():
        out = model.generate(**inputs, **kw)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def ask(question, use_rag=True, use_adapter=True, k=TOP_K, temperature=0.3, show=True):
    ctx = hybrid_search(question, k=k) if use_rag else None
    msgs = build_messages(question, ctx)
    t0 = time.time()
    if use_adapter:
        answer = generate(msgs, temperature=temperature)
    else:
        with model.disable_adapter():
            answer = generate(msgs, temperature=temperature)
    dt = time.time() - t0
    if show:
        print(f"⏱ {dt:.1f}s\n{answer}")
        for i, c in enumerate(ctx or [], 1):
            print(f"   [{i}] {short_ref(c)} · {c['section']} · p.{c['page_start']}")
    return {"answer": answer, "contexts": ctx or [], "latency_s": dt}

q = "Por que a reintrodução de dispersores de sementes é importante para a regeneração da Mata Atlântica?"
for label, rag, ad in [("A · base", False, False), ("B · base + RAG", True, False),
                       ("C · fine-tunado", False, True), ("D · fine-tunado + RAG", True, True)]:
    print(f"\n{'='*20} {label} {'='*20}")
    ask(q, use_rag=rag, use_adapter=ad)

## 3. Avaliação — mesmas 4 configurações, mesmas perguntas, mesmo juiz
O split de teste vem do mesmo arquivo e a amostragem usa a mesma semente: **as perguntas são as mesmas** das rodadas QLoRA e Qwen.

In [ ]:
import anthropic
llm = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def call_llm(system, user, max_tokens=1500, retries=5):
    for attempt in range(retries):
        try:
            r = llm.messages.create(model=GEN_MODEL, max_tokens=max_tokens, system=system,
                                    messages=[{"role": "user", "content": user}])
            return "".join(b.text for b in r.content if b.type == "text")
        except Exception as e:
            wait = 2 ** attempt + random.random()
            print(f"  tentativa {attempt+1} falhou ({type(e).__name__}); aguardando {wait:.0f}s")
            time.sleep(wait)
    return None

def parse_json(txt):
    if not txt:
        return None
    txt = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    a, b = txt.find("{"), txt.rfind("}")
    if a < 0 or b <= a:
        return None
    try:
        return json.loads(txt[a:b + 1])
    except json.JSONDecodeError:
        return None

def retrieval_metrics(items, k=TOP_K):
    hit = doc_hit = 0; rr = 0.0
    for e in items:
        src = chunk_by_id[e["source_chunk_ids"][0]]
        res = hybrid_search(e["question"], k=20)
        pos = next((i for i, c in enumerate(res) if c["doc_id"] == src["doc_id"] and abs(c["idx"] - src["idx"]) <= 1), None)
        hit += pos is not None and pos < k
        rr += 1 / (pos + 1) if pos is not None else 0
        doc_hit += any(c["doc_id"] == src["doc_id"] for c in res[:k])
    n = len(items)
    return {"n": n, f"recall@{k}": hit / n, "mrr@20": rr / n, f"doc_recall@{k}": doc_hit / n}

ret_items = [e for e in test if e["kind"] in ("rag", "unanswerable")]
retrieval_report = retrieval_metrics(ret_items)
print("Recuperação (igual à do Gemma, mesmo índice):", retrieval_report)

In [ ]:
JUDGE_SYSTEM = "Você é um avaliador rigoroso de respostas sobre ecologia e conservação. Responda apenas com JSON."

def judge(question, reference, answer, contexts=None):
    ctx_block = f"\nCONTEXTO DISPONÍVEL AO MODELO:\n{format_context(contexts)}\n" if contexts else ""
    prompt = f"""PERGUNTA: {question}

RESPOSTA DE REFERÊNCIA: {reference}
{ctx_block}
RESPOSTA DO MODELO: {answer}

Avalie:
- "correcao" (1-5): a resposta do modelo transmite o mesmo conteúdo factual da referência? 5 = equivalente e completa; 3 = parcialmente correta; 1 = errada, irrelevante ou em outro idioma.
- "fidelidade" (1-5 ou null): {"tudo o que o modelo afirma está apoiado no CONTEXTO? 5 = nada inventado; 1 = muitas informações sem apoio." if contexts else "use null (não havia contexto)."}
Responda só com JSON: {{"correcao": int, "fidelidade": int|null, "comentario": "uma frase"}}"""
    return parse_json(call_llm(JUDGE_SYSTEM, prompt, max_tokens=300))

N_EVAL = 60
rng = random.Random(SEED)
test_rag = [e for e in test if e["kind"] == "rag"]
test_cb = [e for e in test if e["kind"] == "closed_book"]
test_un = [e for e in test if e["kind"] == "unanswerable"]
eval_items = rng.sample(test_rag, min(len(test_rag), int(N_EVAL * .7))) + rng.sample(test_cb, min(len(test_cb), int(N_EVAL * .3)))
eval_items = [{"question": e["question"], "reference": re.sub(r"\s?\[\d\]", "", e["reference"]),
               "kind": e["kind"], "doc_id": e["doc_id"]} for e in eval_items]
if GOLD_FILE.exists():
    eval_items += [{"question": g["question"], "reference": g["reference_answer"], "kind": "gold",
                    "doc_id": g.get("doc_id", "")} for g in read_jsonl(GOLD_FILE)]
print(f"Itens de avaliação: {len(eval_items)}", dict(Counter(e["kind"] for e in eval_items)))

CONFIGS = {"A: base": (False, False), "B: base+RAG": (True, False), "C: FT": (False, True), "D: FT+RAG": (True, True)}
rows = []
for i, e in enumerate(eval_items, 1):
    ctx = hybrid_search(e["question"])
    for name, (use_rag, use_ad) in CONFIGS.items():
        msgs = build_messages(e["question"], ctx if use_rag else None)
        t0 = time.time()
        if use_ad:
            ans = generate(msgs, temperature=0)
        else:
            with model.disable_adapter():
                ans = generate(msgs, temperature=0)
        lat = time.time() - t0
        j = judge(e["question"], e["reference"], ans, ctx if use_rag else None) or {}
        cites = [int(x) for x in re.findall(r"\[(\d)\]", ans)]
        cite_ok = None
        if use_rag and e["kind"] == "rag":
            cite_ok = bool(cites) and any(1 <= n <= len(ctx) and ctx[n - 1]["doc_id"] == e["doc_id"] for n in cites)
        rows.append({"config": name, "kind": e["kind"], "question": e["question"], "answer": ans,
                     "correcao": j.get("correcao"), "fidelidade": j.get("fidelidade"),
                     "cite_ok": cite_ok, "ptbr": is_ptbr(ans), "latency_s": lat})
    if i % 10 == 0:
        print(f"  {i}/{len(eval_items)}")

df_eval = pd.DataFrame(rows)
df_eval.to_csv(OUT_DIR / RUN_NAME / "eval_detalhado.csv", index=False)

In [ ]:
abst = []
for e in rng.sample(test_un, min(30, len(test_un))) + rng.sample(test_rag, min(30, len(test_rag))):
    ctx = [chunk_by_id[c] for c in e["context_chunk_ids"]]
    msgs = build_messages(e["question"], ctx)
    for name, use_ad in {"B: base+RAG": False, "D: FT+RAG": True}.items():
        if use_ad:
            ans = generate(msgs, temperature=0)
        else:
            with model.disable_adapter():
                ans = generate(msgs, temperature=0)
        abst.append({"config": name, "kind": e["kind"], "abstained": is_abstention(ans)})
df_abst = pd.DataFrame(abst)

summary = df_eval.groupby("config").agg(
    correcao=("correcao", "mean"), fidelidade=("fidelidade", "mean"),
    citacao_correta=("cite_ok", lambda s: s.dropna().astype(float).mean() if s.notna().any() else np.nan),
    ptbr=("ptbr", "mean"), latencia_s=("latency_s", "mean")).round(3)
if len(df_abst):
    summary["abstencao_correta"] = df_abst[df_abst.kind == "unanswerable"].groupby("config").abstained.mean()
    summary["falsa_abstencao"] = df_abst[df_abst.kind == "rag"].groupby("config").abstained.mean()
summary.to_csv(OUT_DIR / RUN_NAME / "eval_resumo.csv")
json.dump({"retrieval": retrieval_report, "summary": json.loads(summary.reset_index().to_json(orient="records"))},
          open(OUT_DIR / RUN_NAME / "eval_resumo.json", "w"), indent=2, ensure_ascii=False)
display(summary)

ax = summary[["correcao", "fidelidade"]].plot.bar(figsize=(8, 4), rot=0, ylim=(0, 5))
ax.set_title("Gemma 4 E2B LoRA 16-bit · Refauna — notas do juiz (1–5)"); ax.grid(axis="y", alpha=.3)
plt.savefig(OUT_DIR / RUN_NAME / "eval_notas.png", dpi=160, bbox_inches="tight"); plt.show()

## 4. Comparação: Gemma QLoRA × Gemma LoRA 16-bit × Qwen LoRA 16-bit
Junta todas as rodadas salvas em `outputs/`. Gera `comparacao_modelos.csv/.png` (atualizados com a nova rodada).

Como ler:
- **Gemma QLoRA × Gemma LoRA** isola o efeito do **método** (mesmo modelo, mesmos dados).
- **Gemma LoRA × Qwen LoRA** isola o efeito do **modelo-base** (mesmo método, mesmos dados).

In [ ]:
import matplotlib.pyplot as plt

rows = []
for rep in sorted(OUT_DIR.glob("*/train_report.json")):
    tr = json.load(open(rep))
    ev = rep.parent / "eval_resumo.csv"
    if not ev.exists():
        continue
    d = pd.read_csv(ev).set_index("config")
    for cfg in d.index:
        rows.append({"modelo": "Qwen3.5-2B" if tr["run"].startswith("qwen") else "Gemma 4 E2B",
                     "método": "QLoRA 4-bit" if tr["load_in_4bit"] else "LoRA 16-bit",
                     "run": tr["run"], "config": cfg, "min_treino": tr["minutes"], "vram_pico_gb": tr["peak_vram_gb"],
                     **d.loc[cfg].to_dict()})
cmp = pd.DataFrame(rows)
cmp.to_csv(OUT_DIR / "comparacao_modelos.csv", index=False)

metricas = [m for m in ["correcao", "fidelidade", "citacao_correta", "abstencao_correta", "falsa_abstencao", "ptbr", "latencia_s"] if m in cmp]
display(cmp.pivot_table(index="config", columns="run", values=metricas).round(2))
display(cmp.drop_duplicates("run")[["run", "modelo", "método", "min_treino", "vram_pico_gb"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axes, ["correcao", "fidelidade"]):
    cmp.pivot_table(index="config", columns="run", values=m).plot.bar(ax=ax, rot=0, ylim=(0, 5))
    ax.set_title(f"{m} (1–5)"); ax.grid(axis="y", alpha=.3); ax.legend(fontsize=8)
plt.suptitle("Refauna · Gemma QLoRA × Gemma LoRA 16-bit × Qwen LoRA 16-bit")
plt.savefig(OUT_DIR / "comparacao_modelos.png", dpi=160, bbox_inches="tight"); plt.show()

## 5. Publicação no Hugging Face (opcional)
Três repositórios **novos** na organização, sem tocar nos do QLoRA. Ficam privados até você revisar.

> Só publique se esta versão for útil para a palestra (ex.: se ela tiver resultado melhor e você quiser usá-la na demo). Os resultados já estão salvos no Drive de qualquer forma.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
for repo in (REPO_LORA, REPO_MERGED, REPO_GGUF):
    api.create_repo(repo, private=HF_PRIVATE, exist_ok=True)

model.push_to_hub(REPO_LORA, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_LORA, token=HF_TOKEN)
print("✅ LoRA:", f"https://huggingface.co/{REPO_LORA}")

In [ ]:
model.push_to_hub_merged(REPO_MERGED, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print("✅ Merged 16-bit:", f"https://huggingface.co/{REPO_MERGED}")

In [ ]:
# Converte com llama.cpp e envia os GGUF (leva alguns minutos)
model.push_to_hub_gguf(REPO_GGUF, tokenizer, quantization_method=["q4_k_m", "q8_0"], token=HF_TOKEN)
print("✅ GGUF:", f"https://huggingface.co/{REPO_GGUF}")

In [ ]:
def eval_table_md():
    try:
        s = json.load(open(OUT_DIR / RUN_NAME / "eval_resumo.json"))
    except FileNotFoundError:
        return "_Avaliação ainda não executada._"
    cols = ["config", "correcao", "fidelidade", "citacao_correta", "abstencao_correta", "ptbr"]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in s["summary"]:
        lines.append("| " + " | ".join("" if r.get(c) is None else (f"{r[c]:.2f}" if isinstance(r[c], float) else str(r[c])) for c in cols) + " |")
    rr = s["retrieval"]
    lines.append(f"\nRecuperação híbrida (teste, n={rr['n']}): Recall@{TOP_K} = {rr[f'recall@{TOP_K}']:.2f}, MRR@20 = {rr['mrr@20']:.2f}")
    return "\n".join(lines)

def model_card(library):
    refs = "\n".join(f"- {d['title']} ({d.get('year') or 's.d.'})" + (f". DOI: [{d['doi']}](https://doi.org/{d['doi']})" if d.get("doi") else "")
                     for d in sorted(docs, key=lambda d: (d.get("year") or 0)))
    return f"""---
license: apache-2.0
base_model: google/gemma-4-e2b-it
language: [pt, en]
library_name: {library}
pipeline_tag: text-generation
tags: [gemma4, unsloth, qlora, lora, rag, refauna, biodiversidade, conservacao, mata-atlantica, refaunacao]
---

# Gemma 4 E2B · Refauna (PT-BR) — LoRA 16-bit

Small Language Model especializado em **defaunação, refaunação, reintrodução de fauna, dispersão de sementes e conservação da Mata Atlântica**, a partir da produção científica do [Refauna](https://refauna.org.br) — iniciativa criada em 2010 para devolver animais silvestres nativos aos seus habitats na Mata Atlântica do Rio de Janeiro.

Variante treinada com **LoRA 16-bit** sobre os mesmos dados da versão QLoRA ([gemma-4-e2b-refauna](https://huggingface.co/{HF_NAMESPACE}/gemma-4-e2b-refauna)), para comparar os dois métodos.

Desenvolvido para a palestra *"Pequenos modelos, conhecimento especializado: fine-tuning e RAG com os dados do Refauna"* (TDC Stage @ Rec'n'Play 2026).

**English:** Gemma 4 E2B fine-tuned (QLoRA, Unsloth) in Brazilian Portuguese to answer questions about Atlantic Forest rewilding, grounded on retrieved passages from Refauna research papers with [n] citations.

## Como foi treinado
- **Base:** `google/gemma-4-e2b-it` (Apache 2.0) via `{BASE_MODEL}`
- **Método:** {"QLoRA 4-bit" if LOAD_IN_4BIT else "LoRA 16-bit"}, r={LORA_R}, alpha={LORA_R}, {EPOCHS} épocas, lr={LR}, só camadas de linguagem, template `gemma-4` (sem thinking)
- **Dados:** {len(train)} exemplos sintéticos em PT-BR gerados a partir de {len(docs)} artigos (estilo RAFT: ~60% com contexto + citações, ~15% abstenção, ~25% sem contexto), com revisão humana amostral e split por documento
- **Hardware:** {train_report['gpu']}, {train_report['minutes']} min, pico de {train_report['peak_vram_gb']} GB de VRAM

## Uso recomendado: com RAG
O modelo foi treinado para receber trechos numerados e citar `[n]`. Formato do prompt:

```
system: {SYSTEM_PROMPT}
user:   CONTEXTO:
        [1] (Autor et al., ano — Título) trecho...
        [2] ...
        PERGUNTA: ...
```

Embeddings usados no RAG: `google/embeddinggemma-300m` + BM25 (fusão RRF), top-{TOP_K}.

### Ollama (Mac/Linux/Windows)
```bash
ollama run hf.co/{REPO_GGUF}:Q4_K_M
```

### Transformers
```python
from transformers import pipeline
pipe = pipeline("text-generation", model="{REPO_MERGED}")
```

## Avaliação
{eval_table_md()}

Notas de 1 a 5 atribuídas por um juiz LLM; A = base, B = base+RAG, C = fine-tunado, D = fine-tunado+RAG.

## Limitações
- Uso educacional e de divulgação científica; **não substitui a literatura original** — sempre consulte as fontes citadas.
- Sem RAG, o modelo pode errar números, datas e nomes de espécies.
- Conhecimento restrito aos artigos usados; não cobre todo o trabalho do Refauna.

## Fontes (atribuição)
{refs}
"""

for repo, lib in [(REPO_LORA, "peft"), (REPO_MERGED, "transformers"), (REPO_GGUF, "gguf")]:
    api.upload_file(path_or_fileobj=model_card(lib).encode("utf-8"), path_in_repo="README.md",
                    repo_id=repo, commit_message="Model card")
print("✅ Model cards enviados")

## 6. No Mac
Se publicar e quiser usar esta versão na demo:
```bash
ollama pull hf.co/Refauna/gemma-4-e2b-refauna-lora16-GGUF:Q4_K_M
```
e acrescente no `config/settings.yaml` do projeto local um modelo `gemma_ft_lora16` com essa tag (mesma família `gemma`). Copie também `outputs/gemma4-e2b-refauna-lora16/` e os novos `comparacao_modelos.*` para `data/outputs/` no Mac.